<a href="https://colab.research.google.com/github/ChinH21204/AI-recruitment/blob/main/DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers pdfplumber openai pandas

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

from openai import OpenAI
client = OpenAI()

In [ ]:
from google.colab import files
uploaded = files.upload()
cv_files = list(uploaded.keys())

In [ ]:
import pdfplumber

def parser_agent(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_agent(cv_text, jd_text):
    emb1 = model.encode(cv_text, convert_to_tensor=True)
    emb2 = model.encode(jd_text, convert_to_tensor=True)
    return float(util.cos_sim(emb1, emb2))

In [ ]:
jd_text = """
Looking for Python Developer with FastAPI, SQL, Machine Learning, 2+ years experience
"""

In [ ]:
def rule_agent(cv_text, similarity):
    skills = ["Python", "FastAPI", "SQL", "Machine Learning"]

    skill_match = sum(1 for s in skills if s.lower() in cv_text.lower()) / len(skills)

    import re
    exp = re.findall(r'(\d+)\s+year', cv_text.lower())
    exp = max([int(x) for x in exp]) if exp else 0
    exp_score = min(exp / 5, 1)

    return 0.5 * similarity + 0.3 * skill_match + 0.2 * exp_score

In [ ]:
def ai_agent(cv_text, jd_text):
    cv_text = cv_text[:1200]

    prompt = f"""
    You are a strict recruiter.

    Evaluate this CV against the job.

    Return JSON:
    - score (0-100)
    - strengths
    - weaknesses
    - recommendation

    CV:
    {cv_text}

    Job:
    {jd_text}
    """

    res = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content

In [ ]:
def scoring_agent(rule_score, use_ai):
    if use_ai:
        return rule_score * 0.7 + 0.3
    return rule_score

In [ ]:
results = []

for file in cv_files:
    text = clean_text(parser_agent(file))

    similarity = embedding_agent(text, jd_text)
    r_score = rule_agent(text, similarity)

    if similarity > 0.3:
        ai_result = ai_agent(text, jd_text)
        final_score = scoring_agent(r_score, True)
    else:
        ai_result = "Rejected early"
        final_score = scoring_agent(r_score, False)

    results.append({
        "cv": file,
        "score": final_score,
        "similarity": similarity,
        "ai": ai_result
    })

In [ ]:
sorted_results = sorted(results, key=lambda x: x["score"], reverse=True)

for i, r in enumerate(sorted_results):
    print(f"\nRank {i+1}")
    print("CV:", r["cv"])
    print("Score:", r["score"])
    print("Similarity:", r["similarity"])
    print("AI:", r["ai"])